# L5 — End-to-End Tracing & Cost Analysis

## What You'll Learn

- How to invoke the Harness programmatically and generate a distributed trace
- How to query X-Ray for trace summaries and display the full span waterfall
- How to read built-in CloudWatch metrics for Bedrock and AgentCore (tokens, latency, invocations)
- How to calculate cost per session from token counts and identify latency bottlenecks by service

## Overview

This notebook follows a single AnyCompany customer request end-to-end through the system — from the Harness through the specialist agents, Gateway, and Lambda tools — and answers three questions: what happened, how long did it take, and what did it cost?

## Architecture: What We're Tracing

```
  User prompt
       │
       ▼
  AgentCore Harness          ← root span
       │  model call
       │  delegate to specialist
       ▼
  AgentCore Gateway          ← child span
       │  route to tool Lambda
       ▼
  Lambda (order / refund)    ← child span
       │  DynamoDB / KB query
       ▼
  Response streamed back
```

Each box becomes a **span** in the X-Ray trace. Steps 3–4 query and display the full span tree.

## Steps

1. Load configuration from SSM
2. Authenticate and invoke the Harness to generate a trace
3. Find the trace in X-Ray
4. Display the span waterfall
5. Query CloudWatch metrics (tokens, latency, invocations)
6. Calculate cost per session
7. Break down latency by service

## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- All L3 notebooks completed — agents deployed, Gateway running, Harness created, and IDs in SSM
- Transaction Search enabled (done in `1_pre-requisites.ipynb` Step 1)
- Tracing enabled for agents, Gateway, and Memory (done in L3 agent notebooks)
- `boto3` and `pandas` (installed in the first code cell)

## Step 1: Load Configuration from SSM

Install required packages.

In [ ]:
%pip install --quiet boto3==1.43.0 pandas==2.2.0

Import libraries, initialise AWS clients, and load workshop configuration from SSM.

In [ ]:
import boto3
import json
import time
import uuid
import os
from datetime import datetime, timedelta, timezone

import pandas as pd

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

ssm = boto3.client("ssm", region_name=REGION)
xray = boto3.client("xray", region_name=REGION)
cloudwatch = boto3.client("cloudwatch", region_name=REGION)
logs_client = boto3.client("logs", region_name=REGION)
cognito = boto3.client("cognito-idp", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)

SSM_PREFIX = "/anycompany/agentcore"
WANTED_KEYS = [
    "region", "account_id", "harness_id", "harness_arn",
    "gateway_id", "gateway_url",
    "cognito_user_pool_id", "cognito_client_id",
    "user_password",  # SecureString — workshop test password
]

cfg = {}
for k in WANTED_KEYS:
    try:
        cfg[k] = ssm.get_parameter(Name="{}/{}".format(SSM_PREFIX, k), WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        cfg[k] = None
        print("  Warning: Missing {}/{}".format(SSM_PREFIX, k))

print("Workshop configuration:")
for k, v in cfg.items():
    print("  {:28} = {}".format(k, v or "(not found)"))

## Step 2: Authenticate and Invoke the Harness

Authenticate as `gold_customer` via Cognito USER_PASSWORD_AUTH.

In [ ]:
# Authenticate as gold_customer
USER_USERNAME = "gold_customer"
# Workshop test password is stored in SSM as a SecureString
# (published by 1_pre-requisites.ipynb). Loaded above with WithDecryption=True.
USER_PASSWORD = cfg.get("user_password") or ""
if not USER_PASSWORD:
    raise RuntimeError(
        "Missing SSM parameter {}/user_password. Run L3 pre-requisites first.".format(SSM_PREFIX)
    )

# Use the cognito_client_id key (L3 pre-reqs writes this)
client_id = cfg.get("cognito_web_client_id")
if not client_id:
    try:
        client_id = ssm.get_parameter(Name="{}/cognito_client_id".format(SSM_PREFIX))["Parameter"]["Value"]
    except Exception:
        raise RuntimeError("Cannot find Cognito client ID in SSM. Run L3 pre-requisites first.")

try:
    auth_resp = cognito.initiate_auth(
        ClientId=client_id,
# PRODUCTION REQUIREMENT: Replace USER_PASSWORD_AUTH with federated identity (SAML/OIDC).
# Shared passwords do not meet enterprise security requirements.
# See: https://docs.aws.amazon.com/cognito/latest/developerguide/cognito-user-pools-identity-federation.html
        AuthFlow="USER_PASSWORD_AUTH",
        AuthParameters={"USERNAME": USER_USERNAME, "PASSWORD": USER_PASSWORD},
    )
    print("Authenticated as {}".format(USER_USERNAME))
except cognito.exceptions.NotAuthorizedException as e:
    print("Authentication failed: {}".format(e))
    print("  Check that test users were created in L3 pre-requisites (Step 3.1)")
    raise
except cognito.exceptions.UserNotFoundException:
    print("User '{}' not found. Run L3 pre-requisites notebook to create test users.".format(USER_USERNAME))
    raise

Set X-Ray sampling to 100%, invoke the Harness with a test order query, and poll until traces appear.

In [ ]:
# Invoke the Harness with a test prompt
SESSION_ID = str(uuid.uuid4()).upper()
ACTOR_ID = "CUST-789"  # gold_customer's customer_id
TEST_PROMPT = "What is the status of order ORD-10001?"

print("Sending: {}".format(TEST_PROMPT))
print("Session: {}".format(SESSION_ID))
print()

if not cfg.get("harness_arn"):
    raise RuntimeError("harness_arn not found in SSM. Run L3 orchestrator notebook first.")

# Ensure X-Ray sampling is set to 100% so traces are captured
try:
    xray.update_indexing_rule(
        Name="Default",
        Rule={"Probabilistic": {"DesiredSamplingPercentage": 100}}
    )
    print("X-Ray sampling set to 100% (ensures traces are captured)")
except Exception as e:
    print("Could not update sampling rate: {}".format(e))

try:
    invoke_start = time.time()
    response = agentcore.invoke_harness(
        harnessArn=cfg["harness_arn"],
        runtimeSessionId=SESSION_ID,
        actorId=ACTOR_ID,
        messages=[{"role": "user", "content": [{"text": "[Authenticated customer: {} | Tier: gold]\n\n{}".format(ACTOR_ID, TEST_PROMPT)}]}],
    )

    full_text = ""
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            text = delta.get("text", "")
            full_text += text

    invoke_latency = time.time() - invoke_start
    print("Response ({:.1f}s):".format(invoke_latency))
    print(full_text[:500])
    if len(full_text) > 500:
        print("...")

except Exception as e:
    print("Harness invocation failed: {}".format(e))
    print("  Ensure L3 notebooks are complete and the Harness is deployed.")
    raise

# Wait for telemetry to propagate, then poll for traces
print("")
print("Waiting for traces to appear in X-Ray (polling every 10s, up to 2 min)...")
TRACE_FOUND = False
for attempt in range(12):
    time.sleep(10)
    check = xray.get_trace_summaries(
        StartTime=datetime.now(timezone.utc) - timedelta(minutes=5),
        EndTime=datetime.now(timezone.utc),
        Sampling=False,
    )
    if check.get("TraceSummaries"):
        TRACE_FOUND = True
        print("Traces available after {}s".format((attempt+1)*10))
        break
    print("  ...{}s - not yet".format((attempt+1)*10))

if not TRACE_FOUND:
    print("No traces appeared after 2 minutes. They may still be propagating.")
    print("  Continue to Step 3 - it will show what is available.")

## Step 3: Find the Trace in X-Ray

Query X-Ray for traces from the last 10 minutes and select the one with the most services (most interesting for the waterfall).

In [ ]:
# Query X-Ray for recent traces
end_time = datetime.now(timezone.utc)
start_time_query = end_time - timedelta(minutes=10)

trace_summaries = xray.get_trace_summaries(
    StartTime=start_time_query,
    EndTime=end_time,
    Sampling=False,
)

traces = trace_summaries.get("TraceSummaries", [])
print("Found {} traces in the last 10 minutes".format(len(traces)))
if len(traces) > 10:
    print("(Showing most recent 10)")
print()

if traces:
    # Display as a table
    trace_data = []
    for t in traces[:10]:
        trace_data.append({
            "Trace ID": t["Id"],
            "Duration (s)": round(t.get("Duration", 0), 2),
            "Response Time (s)": round(t.get("ResponseTime", 0), 2),
            "Has Error": t.get("HasError", False),
            "Has Fault": t.get("HasFault", False),
        })
    df = pd.DataFrame(trace_data)
    print(df.to_string(index=False))

    # Pick the trace with the most services (most interesting for the waterfall)
    best_trace = None
    best_service_count = 0
    for t in traces[:10]:
        detail = xray.batch_get_traces(TraceIds=[t["Id"]])
        segs = detail.get("Traces", [{}])[0].get("Segments", [])
        service_names = set(json.loads(s["Document"]).get("name", "") for s in segs)
        if len(service_names) > best_service_count:
            best_service_count = len(service_names)
            best_trace = t["Id"]

    SELECTED_TRACE_ID = best_trace or traces[0]["Id"]
    print("")
    print("Selected trace (with {} services): {}".format(best_service_count, SELECTED_TRACE_ID))
else:
    SELECTED_TRACE_ID = None
    print("No traces found. Make sure:")
    print("  1. Transaction Search is enabled (L3 pre-requisites Step 2)")
    print("  2. The chatbot was invoked (Step 2 above)")
    print("  3. Wait a few more minutes and re-run this cell")

## Step 4: Display the Span Waterfall

Retrieve the full trace, parse all segments and subsegments into a flat span list, and print the indented waterfall. Red spans indicate errors.

In [ ]:
# Retrieve full trace detail
if not SELECTED_TRACE_ID:
    print("No trace selected. Re-run Step 3 after waiting a few more minutes.")
else:
    trace_detail = xray.batch_get_traces(TraceIds=[SELECTED_TRACE_ID])
    segments_raw = trace_detail.get("Traces", [{}])[0].get("Segments", [])

    if not segments_raw:
        print("Trace {} has no segments yet. Wait and retry.".format(SELECTED_TRACE_ID))
    else:
        # Parse segments into a flat list of spans
        all_spans = []

        def parse_segment(doc, parent_id=None, depth=0):
            span = {
                "name": doc.get("name", "unknown"),
                "id": doc.get("id", ""),
                "parent_id": parent_id,
                "depth": depth,
                "start": doc.get("start_time", 0),
                "end": doc.get("end_time", 0),
                "duration_ms": round((doc.get("end_time", 0) - doc.get("start_time", 0)) * 1000, 1),
                "error": doc.get("error", False),
                "fault": doc.get("fault", False),
            }
            all_spans.append(span)
            for sub in doc.get("subsegments", []):
                parse_segment(sub, parent_id=doc.get("id"), depth=depth + 1)

        for seg in segments_raw:
            doc = json.loads(seg["Document"])
            parse_segment(doc)

        # Display as an indented waterfall
        print("Trace: {}".format(SELECTED_TRACE_ID))
        print("Total spans: {}".format(len(all_spans)))
        print("=" * 80)
        print("{:<50} {:>10} {:>8}".format("Span", "Duration", "Status"))
        print("-" * 80)

        for span in all_spans:
            indent = "  " * span["depth"]
            prefix = "+-- " if span["depth"] > 0 else ""
            name = "{}{}{}".format(indent, prefix, span["name"])
            duration = "{:.0f}ms".format(span["duration_ms"])
            status = "ERR" if span["error"] or span["fault"] else "ok"
            print("{:<50} {:>10} {:>8}".format(name, duration, status))

        print("=" * 80)

## Step 5: Query CloudWatch Metrics

Query the last hour of Bedrock model metrics (tokens, latency, invocations) and AgentCore metrics (Gateway invocations, Memory events).

In [ ]:
# Query CloudWatch metrics for the last hour
end_time_cw = datetime.now(timezone.utc)
start_time_cw = end_time_cw - timedelta(hours=1)

print("CloudWatch Metrics - Last 1 hour")
print("=" * 60)

# Bedrock model metrics (work without dimensions)
print("")
print("-- AWS/Bedrock (Model Usage) --")
bedrock_metrics = [
    ("InputTokenCount", "Sum"),
    ("OutputTokenCount", "Sum"),
    ("InvocationLatency", "Average"),
    ("Invocations", "Sum"),
]
for metric_name, stat in bedrock_metrics:
    try:
        resp = cloudwatch.get_metric_statistics(
            Namespace="AWS/Bedrock",
            MetricName=metric_name,
            StartTime=start_time_cw,
            EndTime=end_time_cw,
            Period=3600,
            Statistics=[stat],
        )
        datapoints = resp.get("Datapoints", [])
        if datapoints:
            value = sum(dp.get(stat, 0) for dp in datapoints)
            unit = "ms" if "Latency" in metric_name else ""
            print("  {:25} = {:,.0f}{}".format(metric_name, value, unit))
        else:
            print("  {:25} = (no data)".format(metric_name))
    except Exception as e:
        print("  {:25} = Error: {}".format(metric_name, e))

# AgentCore Gateway metrics (with dimensions)
print("")
print("-- AWS/Bedrock-AgentCore (Gateway + Memory) --")
try:
    resp = cloudwatch.get_metric_statistics(
        Namespace="AWS/Bedrock-AgentCore",
        MetricName="Invocations",
        Dimensions=[{"Name": "Operation", "Value": "InvokeGateway"}],
        StartTime=start_time_cw, EndTime=end_time_cw,
        Period=3600, Statistics=["Sum"],
    )
    datapoints = resp.get("Datapoints", [])
    value = sum(dp.get("Sum", 0) for dp in datapoints) if datapoints else 0
    print("  Gateway Invocations      = {:.0f}".format(value))
except Exception as e:
    print("  Gateway Invocations      = Error: {}".format(e))

try:
    resp = cloudwatch.get_metric_statistics(
        Namespace="AWS/Bedrock-AgentCore",
        MetricName="Invocations",
        Dimensions=[{"Name": "Operation", "Value": "CreateEvent"}],
        StartTime=start_time_cw, EndTime=end_time_cw,
        Period=3600, Statistics=["Sum"],
    )
    datapoints = resp.get("Datapoints", [])
    value = sum(dp.get("Sum", 0) for dp in datapoints) if datapoints else 0
    print("  Memory CreateEvent       = {:.0f}".format(value))
except Exception as e:
    print("  Memory CreateEvent       = Error: {}".format(e))

print("")
print("=" * 60)

## Step 6: Calculate Cost Per Session

Pull input and output token counts from CloudWatch and multiply by Claude Sonnet 4 pricing ($3.00/M input, $15.00/M output) to estimate session cost.

In [ ]:
# Calculate cost from CloudWatch Bedrock metrics
print("Calculating cost...")
BEDROCK_NAMESPACE = "AWS/Bedrock"
cost_start = datetime.now(timezone.utc) - timedelta(hours=1)
cost_end = datetime.now(timezone.utc)

resp_in = cloudwatch.get_metric_statistics(
    Namespace=BEDROCK_NAMESPACE, MetricName="InputTokenCount",
    StartTime=cost_start, EndTime=cost_end, Period=3600, Statistics=["Sum"],
)
resp_out = cloudwatch.get_metric_statistics(
    Namespace=BEDROCK_NAMESPACE, MetricName="OutputTokenCount",
    StartTime=cost_start, EndTime=cost_end, Period=3600, Statistics=["Sum"],
)

total_input_tokens = int(sum(dp.get("Sum", 0) for dp in resp_in.get("Datapoints", [])))
total_output_tokens = int(sum(dp.get("Sum", 0) for dp in resp_out.get("Datapoints", [])))

INPUT_PRICE_PER_M = 3.00
OUTPUT_PRICE_PER_M = 15.00
input_cost = (total_input_tokens / 1_000_000) * INPUT_PRICE_PER_M
output_cost = (total_output_tokens / 1_000_000) * OUTPUT_PRICE_PER_M
total_cost = input_cost + output_cost

print("Cost Estimate (Last 1 Hour)")
print("=" * 50)
print("  Input tokens:  {:>10,}  (${:.6f})".format(total_input_tokens, input_cost))
print("  Output tokens: {:>10,}  (${:.6f})".format(total_output_tokens, output_cost))
print("  " + "-" * 40)
print("  Total cost:                ${:.6f}".format(total_cost))
print()
if total_input_tokens == 0 and total_output_tokens == 0:
    print("  No token data yet. Metrics may take a few minutes to appear.")
    print("  Try re-running this cell after 5 minutes.")
else:
    print("  At this rate, 1000 sessions would cost ~${:.2f}".format(total_cost * 1000))

## Step 7: Latency Breakdown by Service

Aggregate span durations by service name and print sorted by time spent. Model-bound = try a faster model. Tool-bound = optimize Lambda cold starts or cache DynamoDB results.

In [ ]:
# Latency breakdown by service
if not all_spans:
    print("No spans available. Run Steps 3-4 first.")
else:
    from collections import defaultdict
    service_times = defaultdict(float)
    total_time_ms = 0

    for span in all_spans:
        service_times[span["name"]] += span["duration_ms"]
        total_time_ms += span["duration_ms"]

    print("Latency Breakdown by Service")
    print("=" * 60)

    for name, duration in sorted(service_times.items(), key=lambda x: -x[1]):
        pct_val = "{:.0f}%".format(duration/total_time_ms*100) if total_time_ms > 0 else "N/A"
        print("  {:45} {:>8.0f}ms  ({})".format(name, duration, pct_val))

    print("  " + "-" * 55)
    print("  {:45} {:>8.0f}ms".format("TOTAL", total_time_ms))
    print()
    print("  Each row is a service that participated in this trace.")
    print("  Gateway + Agent spans = tool call time. Memory spans = context loading.")

## Summary

| What We Did | How |
|---|---|
| Generated a traced request | `invoke_harness` with a test prompt |
| Found the trace | `xray.get_trace_summaries` |
| Displayed the span tree | `xray.batch_get_traces` -> parse segments |
| Queried built-in metrics | `cloudwatch.get_metric_statistics` |
| Calculated cost | Token counts x model pricing |
| Identified bottlenecks | Latency breakdown by service |
